# HireFlow 系统评估

Run All Cells → 运行 Pipeline → 计算指标 → 生成报告 → 对比历史

In [1]:
# ================================================================
# Cell 1: 初始化
# ================================================================
import sys, os, time, json, re, glob, math
from datetime import datetime
import pytz

# 修复路径
project_root = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == "evaluation" else os.getcwd()
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# 时间戳
sydney_tz = pytz.timezone("Australia/Sydney")
now = datetime.now(sydney_tz)
report_time = now.strftime("%I-%M-%p")
display_time = now.strftime("%I:%M %p")
report_date = now.strftime("%Y-%m-%d")
report_filename = f"{report_date}-{report_time}"

reports_dir = os.path.join(os.getcwd(), "reports")
os.makedirs(reports_dir, exist_ok=True)

report_md = os.path.join(reports_dir, f"{report_filename}.md")
report_json = os.path.join(reports_dir, f"{report_filename}.json")

print(f"报告: reports/{report_filename}.md")
print(f"指标: reports/{report_filename}.json")
print(f"时间: {report_date} {display_time}")

报告: reports/2026-05-31-04-34-PM.md
指标: reports/2026-05-31-04-34-PM.json
时间: 2026-05-31 04:34 PM


In [2]:
# ================================================================
# Cell 2: 健康检查
# ================================================================
from app.utils.config import settings
from openai import OpenAI

c = OpenAI(base_url=settings.llm.local_base_url, api_key=settings.llm.local_api_key)
assert settings.llm.local_model in [m.id for m in c.models.list().data], f"LLM 未找到"

c = OpenAI(base_url=settings.embedding.local_base_url, api_key=settings.embedding.local_api_key)
emb_dim = len(c.embeddings.create(model=settings.embedding.local_model, input="t").data[0].embedding)

from app.database.session import init_db; init_db()
from qdrant_client import QdrantClient; QdrantClient(url=settings.qdrant.url).get_collections()

print(f"健康检查通过 | {settings.llm.local_model} | embedding({emb_dim}维) | mode={settings.llm.mode}")

健康检查通过 | hermes-3-llama-3.1-8b | embedding(2560维) | mode=local


In [3]:
# ================================================================
# Cell 3: 运行 Pipeline
# ================================================================
from app.agents.jd_agent import analyze_jd
from app.agents.resume_agent import batch_parse_resumes
from app.agents.match_agent import batch_match_candidates
from app.agents.ranking_agent import rank_candidates

TEST_JD = """
岗位名称: Python 后端开发工程师
必备技能: Python, FastAPI, PostgreSQL, Docker, Git
加分技能: LangChain, RAG, Redis
岗位职责: 开发后端API, 数据库设计, 编写单元测试
学历要求: 计算机相关专业本科及以上
经验要求: 0-3年
"""

TEST_RESUMES = {
    "E001": "姓名: 张工\\n技能: Python, FastAPI, PostgreSQL, Docker, Git, Redis\\n项目: 电商API\\n教育: 2020-2024 北大 CS学士\\n经历: 2023 Python实习生",
    "E002": "姓名: 李工\\n技能: Python, Django, MySQL, Docker, Git\\n项目: 博客系统\\n教育: 2019-2023 浙大 SE学士\\n经历: 2022 Django实习生",
    "E003": "姓名: 王工\\n技能: Python, FastAPI, PostgreSQL, LangChain, RAG, Docker, Git, Redis\\n项目: RAG问答系统\\n教育: 2021-2023 清华 AI硕士\\n经历: 2023 AI后端实习生",
}

print("运行 Pipeline...\n")
timeline = []
total_start = time.time()

# [1/4] JD
t0 = time.time()
jd_profile = await analyze_jd(TEST_JD)
t = time.time() - t0
timeline.append(("JD 解析", t))
print(f"  [1] JD解析: {t:.1f}s -> {jd_profile.get('job_title')}")

# [2/4] Resume
t0 = time.time()
profiles = await batch_parse_resumes(TEST_RESUMES)
t = time.time() - t0
timeline.append(("简历解析", t))
ids = list(TEST_RESUMES.keys())
for i, p in enumerate(profiles):
    p["candidate_id"] = ids[i]
print(f"  [2] 简历解析: {t:.1f}s ({len(profiles)}份)")

# [3/4] Match
t0 = time.time()
rubric = jd_profile.pop("rubric", None)
matches = await batch_match_candidates(jd_profile, profiles, rubric=rubric)
t = time.time() - t0
timeline.append(("匹配评分", t))
print(f"  [3] 匹配评分: {t:.1f}s")

# [4/4] Rank
t0 = time.time()
ranking = await rank_candidates(matches)
t = time.time() - t0
timeline.append(("排序", t))
print(f"  [4] 排序: {t:.1f}s")

total_time = time.time() - total_start
ranked = ranking.get("ranked_candidates", [])
summary = ranking.get("summary", {})
scores = [c.get("total_score", 0) for c in ranked]

print(f"\\nPipeline 完成 | 总耗时: {total_time:.1f}s\\n")

运行 Pipeline...



  [1] JD解析: 3.0s -> Python 后端开发工程师


  [2] 简历解析: 19.0s (3份)


  [3] 匹配评分: 25.4s


  [4] 排序: 2.4s
\nPipeline 完成 | 总耗时: 49.7s\n


In [4]:
# ================================================================
# Cell 4: 计算指标 + 生成报告
# ================================================================

# ---- 代理 Ground Truth (关键词匹配) ----
jd_skills = jd_profile.get("required_skills", [])

relevance = {}
for cid, text in TEST_RESUMES.items():
    hits = sum(1 for s in jd_skills if s.lower() in text.lower())
    relevance[cid] = hits / max(len(jd_skills), 1)

ranked_ids = [c.get("candidate_id", "?") for c in ranked]

# ---- 评估指标 ----
def precision_at_k(ranked_ids, relevance, k, threshold=0.5):
    top_k = ranked_ids[:k]
    if not top_k: return 0.0
    return sum(1 for cid in top_k if relevance.get(cid, 0) >= threshold) / k

def ndcg_at_k(ranked_ids, relevance, k):
    if not ranked_ids: return 0.0
    dcg = sum(relevance.get(cid, 0) / math.log2(i+2) for i, cid in enumerate(ranked_ids[:k]))
    ideal = sorted(relevance.keys(), key=lambda x: relevance.get(x, 0), reverse=True)
    idcg = sum(relevance.get(cid, 0) / math.log2(i+2) for i, cid in enumerate(ideal[:k]))
    return dcg / idcg if idcg > 0 else 0.0

def spearman_rho(ranked_ids, relevance):
    ideal = sorted(relevance.keys(), key=lambda x: relevance.get(x, 0), reverse=True)
    sr = {cid: i for i, cid in enumerate(ranked_ids)}
    ir = {cid: i for i, cid in enumerate(ideal)}
    n = len(ranked_ids)
    if n < 2: return 0.0
    d2 = sum((sr.get(cid, n) - ir.get(cid, n))**2 for cid in ranked_ids)
    return 1 - (6 * d2) / (n * (n*n - 1))

metrics = {}
for k_val in [1, 2, 3]:
    metrics[f"precision_at_{k_val}"] = round(precision_at_k(ranked_ids, relevance, k_val), 3)
metrics["ndcg_at_3"] = round(ndcg_at_k(ranked_ids, relevance, 3), 3)
metrics["spearman_rho"] = round(spearman_rho(ranked_ids, relevance), 3)
if scores:
    metrics["score_max"] = max(scores)
    metrics["score_min"] = min(scores)
    metrics["score_mean"] = round(sum(scores)/len(scores), 1)
    metrics["score_range"] = max(scores) - min(scores)

# ---- 写 JSON ----
eval_data = {
    "timestamp": datetime.now(sydney_tz).isoformat(),
    "pipeline": {"steps": [{"step": n, "time": round(t, 2)} for n,t in timeline], "total_time": round(total_time, 2)},
    "ranking": ranked_ids,
    "scores": [{"id": c.get("candidate_id"), "score": c.get("total_score",0), "rec": c.get("recommendation","")} for c in ranked],
    "metrics": metrics,
    "proxy_relevance": {k: round(v, 2) for k, v in relevance.items()},
}
with open(report_json, "w", encoding="utf-8") as f:
    json.dump(eval_data, f, ensure_ascii=False, indent=2)

# ---- 写 Markdown 报告 ----
md_lines = []
w = md_lines.append
w(f"# HireFlow Pipeline 评估报告")
w(f"")
w(f"**日期:** {report_date}  **时间:** {display_time}  ")
w(f"**LLM:** {settings.llm.mode} ({settings.llm.local_model if settings.llm.mode == 'local' else settings.llm.cloud_model})  ")
w(f"")
w(f"## 一、Pipeline 性能")
w(f"")
w(f"| 步骤 | 耗时 | 占比 |")
w(f"|------|------|------|")
for name, t_val in timeline:
    pct = t_val / total_time * 100 if total_time > 0 else 0
    w(f"| {name} | {t_val:.1f}s | {pct:.0f}% |")
w(f"| **总计** | **{total_time:.1f}s** | **100%** |")
w(f"")
w(f"## 二、评估指标")
w(f"")
w(f"| 指标 | 值 | 说明 |")
w(f"|------|----|------|")
for k_idx in [1, 2, 3]:
    if f"precision_at_{k_idx}" in metrics:
        w(f"| Precision@{k_idx} | {metrics[f'precision_at_{k_idx}']:.3f} | Top {k_idx} 中真正的相关比例 |")
w(f"| NDCG@3 | {metrics.get('ndcg_at_3', 0):.3f} | 排序质量 (1=完美) |")
w(f"| Spearman ρ | {metrics.get('spearman_rho', 0):.3f} | 排序相关性 (-1~1) |")
w(f"")
w(f"*代理 Ground Truth: JD 关键词在简历中的命中率 (需人工标注替换)*")
w(f"")
w(f"## 三、候选人排序")
w(f"")
w(f"| 排名 | 候选人 | 总分 | 技术 | 项目 | 经验 | 教育 | 领域 | 沟通 | 风险 | 等级 |")
w(f"|------|--------|------|------|------|------|------|------|------|------|------|")
for i, c in enumerate(ranked):
    score = c.get("total_score", 0)
    cid = c.get("candidate_id", "?")
    rec = c.get("recommendation", "")
    dims = c.get("dimension_scores", {})
    if isinstance(dims, dict):
        vals = [dims.get(k, "-") for k in ["technical_skills","project_relevance","experience","education","domain_relevance","communication","risk_penalty"]]
    else:
        vals = ["-"] * 7
    w(f"| {i+1} | {cid} | {score:.0f} | {vals[0]} | {vals[1]} | {vals[2]} | {vals[3]} | {vals[4]} | {vals[5]} | {vals[6]} | {rec} |")
w(f"")
w(f"## 四、分数分布")
w(f"")
w(f"| 指标 | 值 |")
w(f"|------|----|")
for key, label in [("score_max","最高分"),("score_min","最低分"),("score_mean","平均分"),("score_range","极差")]:
    if key in metrics:
        w(f"| {label} | {metrics[key]} |")
w(f"")
w(f"---")
w(f"*自动生成 @ {display_time}*")

with open(report_md, "w", encoding="utf-8") as f:
    f.write("\n".join(md_lines))

# ---- 终端输出 ----
print(f"报告已生成: reports/{report_filename}.md + .json")
print(f"\n指标:")
for key in sorted(metrics.keys()):
    print(f"  {key}: {metrics[key]}")
print()
for line in md_lines:
    print(line)

报告已生成: reports/2026-05-31-04-34-PM.md + .json

指标:
  ndcg_at_3: 1.0
  precision_at_1: 1.0
  precision_at_2: 1.0
  precision_at_3: 1.0
  score_max: 85.0
  score_mean: 81.7
  score_min: 75.0
  score_range: 10.0
  spearman_rho: 1.0

# HireFlow Pipeline 评估报告

**日期:** 2026-05-31  **时间:** 04:34 PM  
**LLM:** local (hermes-3-llama-3.1-8b)  

## 一、Pipeline 性能

| 步骤 | 耗时 | 占比 |
|------|------|------|
| JD 解析 | 3.0s | 6% |
| 简历解析 | 19.0s | 38% |
| 匹配评分 | 25.4s | 51% |
| 排序 | 2.4s | 5% |
| **总计** | **49.7s** | **100%** |

## 二、评估指标

| 指标 | 值 | 说明 |
|------|----|------|
| Precision@1 | 1.000 | Top 1 中真正的相关比例 |
| Precision@2 | 1.000 | Top 2 中真正的相关比例 |
| Precision@3 | 1.000 | Top 3 中真正的相关比例 |
| NDCG@3 | 1.000 | 排序质量 (1=完美) |
| Spearman ρ | 1.000 | 排序相关性 (-1~1) |

*代理 Ground Truth: JD 关键词在简历中的命中率 (需人工标注替换)*

## 三、候选人排序

| 排名 | 候选人 | 总分 | 技术 | 项目 | 经验 | 教育 | 领域 | 沟通 | 风险 | 等级 |
|------|--------|------|------|------|------|------|------|------|------|------|
| 1 | E001 | 85 | 30.0 | 20.0 | 15.0 | 10.0 

In [5]:
# ================================================================
# Cell 5: 历史对比 (最近 5 次)
# ================================================================
# 扫描 reports/ 中所有 JSON 文件，提取最近 5 次的指标进行对比

json_files = sorted(glob.glob(os.path.join(reports_dir, "*.json")), reverse=True)

if len(json_files) == 0:
    print("暂无历史数据 (这是第一次运行)")
elif len(json_files) == 1:
    print("仅 1 次记录，无法对比。再跑几次后这里会显示趋势。")
else:
    # 取最近 5 次
    recent = json_files[:5]
    records = []
    for fpath in recent:
        with open(fpath, "r") as f:
            records.append(json.load(f))

    print(f"历史对比 (最近 {len(records)} 次, 共 {len(json_files)} 次记录)\\n")

    # ---- 表1: Pipeline 耗时对比 ----
    print("【Pipeline 总耗时对比】")
    print(f"  {'时间':<24} {'总耗时':<10} {'变化':<10}")
    print(f"  {'-'*44}")
    for i, rec in enumerate(records):
        ts = rec.get("timestamp", "?")[:16].replace("T", " ")
        total = rec["pipeline"]["total_time"]
        if i == 0:
            delta = "(当前)"
        else:
            prev = records[i-1]["pipeline"]["total_time"]
            diff = total - prev
            if abs(diff) < 1:
                delta = "持平"
            elif diff < 0:
                delta = f"↓ {abs(diff):.1f}s (更快)"
            else:
                delta = f"↑ {diff:.1f}s (更慢)"
        print(f"  {ts:<24} {total:<10.1f}s {delta:<15}")

    # ---- 表2: 指标对比 ----
    metric_keys = ["precision_at_1", "precision_at_3", "ndcg_at_3", "spearman_rho", "score_mean", "score_range"]
    metric_labels = ["P@1", "P@3", "NDCG@3", "Spearman ρ", "平均分", "极差"]

    print(f"\\n【评估指标对比】")
    header = f"  {'时间':<18}"
    for label in metric_labels:
        header += f" {label:>8}"
    print(header)
    print(f"  {'-'*(18 + 8*len(metric_labels))}")

    for rec in records:
        ts = rec.get("timestamp", "?")[:16].replace("T", " ")
        ts_short = ts[-14:] if len(ts) > 14 else ts
        row = f"  {ts_short:<18}"
        m = rec.get("metrics", {})
        for key in metric_keys:
            if key in m:
                row += f" {m[key]:>8.3f}"
            else:
                row += f" {'-':>8}"
        print(row)

    # ---- 表3: 排名一致性 ----
    if len(records) >= 2:
        print(f"\\n【排名一致性】")
        for i, rec in enumerate(records):
            ts = rec.get("timestamp", "?")[:16].replace("T", " ")
            order = " > ".join(rec.get("ranking", []))
            tag = "(当前)" if i == 0 else ""
            print(f"  {ts:<18} {order} {tag}")

    # ---- 变化总结 ----
    if len(records) >= 2:
        newest = records[0]["metrics"]
        oldest = records[-1]["metrics"]
        print(f"\\n【变化趋势 (最新 vs 最旧)】")
        for key, label in zip(metric_keys, metric_labels):
            if key in newest and key in oldest:
                diff = newest[key] - oldest[key]
                if abs(diff) < 0.01:
                    trend = "→ 持平"
                elif diff > 0:
                    trend = f"↗ +{diff:.3f}"
                else:
                    trend = f"↘ {diff:.3f}"
                print(f"  {label:<12} {oldest[key]:.3f} → {newest[key]:.3f}  {trend}")

历史对比 (最近 2 次, 共 2 次记录)\n
【Pipeline 总耗时对比】
  时间                       总耗时        变化        
  --------------------------------------------
  2026-05-31 16:35         49.7      s (当前)           
  2026-05-31 16:33         47.3      s ↓ 2.4s (更快)    
\n【评估指标对比】
  时间                      P@1      P@3   NDCG@3 Spearman ρ      平均分       极差
  ------------------------------------------------------------------
  26-05-31 16:35        1.000    1.000    1.000    1.000   81.700   10.000
  26-05-31 16:33        1.000    1.000    1.000    1.000   81.300   11.000
\n【排名一致性】
  2026-05-31 16:35   E001 > E003 > E002 (当前)
  2026-05-31 16:33   E001 > E003 > E002 
\n【变化趋势 (最新 vs 最旧)】
  P@1          1.000 → 1.000  → 持平
  P@3          1.000 → 1.000  → 持平
  NDCG@3       1.000 → 1.000  → 持平
  Spearman ρ   1.000 → 1.000  → 持平
  平均分          81.300 → 81.700  ↗ +0.400
  极差           11.000 → 10.000  ↘ -1.000


### 使用说明

**前置条件:**
```bash
# 终端 1: 数据库
docker compose up -d postgres qdrant

# 终端 2 / LM Studio 桌面应用: 加载模型
#   hermes-3-llama-3.1-8b
#   text-embedding-qwen3-embedding-4b
```

**运行方式:**
```bash
conda activate hireflowagents
cd evaluation
jupyter notebook 系统评估报告.ipynb
# 或: VS Code 打开 .ipynb → Run All
# 或: python run_eval.py  (仅脚本, 无对比功能)
```

**输出文件:**
```
evaluation/reports/
├── 2026-05-31-04-12-PM.md    # Markdown 报告
├── 2026-05-31-04-12-PM.json  # 结构化指标 (程序化对比用)
└── ...
```